# Regime-Aware ML Trading System
## Full Project Walkthrough & Visualization

This notebook contains the **entire project** — data loading, pattern detection, testing, and visualization.

**How to use:** Run cells top to bottom (`Shift+Enter` or `Runtime > Run all`).

---
## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
# Clone the GitHub repo
!git clone https://github.com/zaetae/regime-aware-ml-trading.git

# Move into the project directory
import os
os.chdir('regime-aware-ml-trading/regime-aware-ml-trading')
print('Working directory:', os.getcwd())

In [ ]:
# Install dependencies
!pip install -q yfinance hmmlearn reportlab

In [ ]:
# Add project root to Python path
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Better plots in Colab
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Setup complete!')

---
## 2. Download & Load Data

In [ ]:
# Download SPY data from Yahoo Finance
from src.data.download_data import download_spy

df = download_spy()
print(f'Downloaded {len(df)} rows')

In [ ]:
# Load and inspect
from src.data.load_data import load_spy

df = load_spy()
print(f'Loaded: {len(df)} bars from {df.index[0].date()} to {df.index[-1].date()}')
print(f'Columns: {list(df.columns)}')
print()
df.head(10)

In [ ]:
# Quick stats
df.describe().round(2)

In [ ]:
# Visualize raw data
fig, axes = plt.subplots(3, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1, 1]})

axes[0].plot(df.index, df['Close'], color='#333', lw=0.7)
axes[0].set_title('SPY Daily Close Price', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Price ($)')

axes[1].bar(df.index, df['Volume']/1e6, width=2, color='#78909C', alpha=0.6)
axes[1].set_ylabel('Volume (M)')

daily_returns = df['Close'].pct_change() * 100
axes[2].fill_between(df.index, daily_returns, alpha=0.5, color='#5C6BC0')
axes[2].set_ylabel('Daily Return (%)')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.show()

---
## 3. ATR — The Volatility Foundation

**Average True Range (ATR)** is the backbone of all our thresholds. Every proximity band, breakout threshold, and validation check is expressed as a multiple of ATR so it adapts to volatility automatically.

In [ ]:
from src.data.utils import compute_atr

atr = compute_atr(df, window=14)

print(f'ATR(14) stats:')
print(f'  Mean:    ${atr.mean():.2f}')
print(f'  Min:     ${atr.min():.2f}')
print(f'  Max:     ${atr.max():.2f}')
print(f'  Current: ${atr.iloc[-1]:.2f}')

In [ ]:
# ATR over time
fig, axes = plt.subplots(2, 1, figsize=(14, 6), gridspec_kw={'height_ratios': [2, 1.5]})

axes[0].plot(df.index, df['Close'], color='#333', lw=0.7)
axes[0].set_title('SPY Price vs ATR(14)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Price ($)')

axes[1].fill_between(df.index, atr, alpha=0.4, color='#E91E63')
axes[1].plot(df.index, atr, color='#E91E63', lw=0.8)
axes[1].set_ylabel('ATR(14) ($)')
axes[1].set_xlabel('Date')
axes[1].set_title('Notice how ATR spikes during COVID (2020) and normalizes after', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Zoomed: show how TR components work
slc = df.iloc[2000:2100]
atr_slc = atr.iloc[2000:2100]
prev_close = slc['Close'].shift(1)
tr1 = slc['High'] - slc['Low']
tr2 = (slc['High'] - prev_close).abs()
tr3 = (slc['Low'] - prev_close).abs()
tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(slc.index, tr, width=1.5, alpha=0.3, color='#FF9800', label='True Range (per bar)')
ax.plot(slc.index, atr_slc, color='#E91E63', lw=2, label='ATR(14) = 14-bar SMA of TR')
ax.set_title('True Range vs ATR — Zoomed 100-Bar Window', fontsize=12, fontweight='bold')
ax.set_ylabel('$ value')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 4. Pattern Detectors — Run All Four

Each detector returns a DataFrame with pattern columns added. Let's run them all and see the results.

In [ ]:
from src.patterns.support_resistance import calculate_support_resistance
from src.patterns.triangles import detect_triangle_pattern
from src.patterns.channels import detect_channel
from src.patterns.multiple_tops_bottoms import detect_multiple_tops_bottoms

print('Running detectors...')

sr_df = calculate_support_resistance(df)
sr = sr_df['near_support'] | sr_df['near_resistance']
print(f'  Support/Resistance: {sr.sum()} events ({sr.mean()*100:.1f}%)')

tri_df = detect_triangle_pattern(df)
tri = tri_df['triangle_pattern'].notna()
print(f'  Triangles:          {tri.sum()} events ({tri.mean()*100:.1f}%)')

ch_df = detect_channel(df)
ch = ch_df['channel_pattern'].notna()
print(f'  Channels:           {ch.sum()} events ({ch.mean()*100:.1f}%)')

mtb_df = detect_multiple_tops_bottoms(df)
mtb = mtb_df['multiple_top_bottom_pattern'].notna()
print(f'  Multi Top/Bottom:   {mtb.sum()} events ({mtb.mean()*100:.1f}%)')

combined = sr | tri | ch | mtb
print(f'\n  COMBINED:           {combined.sum()} events ({combined.mean()*100:.1f}%)')
print(f'  Target range:       {400/len(df)*100:.1f}% - {1200/len(df)*100:.1f}%')

### 4a. Support & Resistance — Visualized

In [ ]:
# Pick a 200-bar window to visualize S/R
start, end = 1500, 1700
slc = sr_df.iloc[start:end]
atr_slc = atr.iloc[start:end]
band = 0.3 * atr_slc

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(slc.index, slc['Close'], color='#333', lw=0.8, label='Close')
ax.plot(slc.index, slc['resistance'], color='#F44336', lw=1, ls='--', label='Resistance (50-bar max High)')
ax.plot(slc.index, slc['support'], color='#2196F3', lw=1, ls='--', label='Support (50-bar min Low)')

ax.fill_between(slc.index, slc['resistance'] - band, slc['resistance'] + band, alpha=0.1, color='#F44336')
ax.fill_between(slc.index, slc['support'] - band, slc['support'] + band, alpha=0.1, color='#2196F3', label='0.3x ATR band')

near = slc[slc['near_support'] | slc['near_resistance']]
ax.scatter(near.index, near['Close'], color='#E91E63', s=20, zorder=5, label=f'Events ({len(near)} in window)')

ax.set_title('Support & Resistance with ATR-Adaptive Proximity Bands', fontsize=13, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

### 4b. Triangle Breakouts — Visualized

In [ ]:
# Find triangle events and visualize one
tri_events = tri_df[tri_df['triangle_pattern'].notna()]
print(f'Total triangle breakouts: {len(tri_events)}')
print(f'\nBreakdown:')
print(tri_events['triangle_pattern'].value_counts())

if len(tri_events) > 0:
    # Pick one from the middle
    ev = tri_events.iloc[len(tri_events) // 2]
    ev_idx = df.index.get_loc(ev.name)
    lo, hi = max(0, ev_idx - 60), min(len(df), ev_idx + 15)
    slc = df.iloc[lo:hi]

    fig, ax = plt.subplots(figsize=(14, 6))
    for _, row in slc.iterrows():
        c = '#26a69a' if row['Close'] >= row['Open'] else '#ef5350'
        ax.plot([row.name, row.name], [row['Low'], row['High']], color=c, lw=0.7)
        ax.plot([row.name, row.name], [row['Open'], row['Close']], color=c, lw=3)

    # Draw the trendlines
    win = 50
    wslc = df.iloc[ev_idx - win:ev_idx]
    x = np.arange(win)
    hc = np.polyfit(x, wslc['High'].values, 1)
    lc = np.polyfit(x, wslc['Low'].values, 1)
    ax.plot(wslc.index, np.polyval(hc, x), color='#F44336', lw=1.5, ls=':', label='High trendline')
    ax.plot(wslc.index, np.polyval(lc, x), color='#2196F3', lw=1.5, ls=':', label='Low trendline')

    ax.axvline(ev.name, color='#E91E63', lw=2, ls='--', label='Breakout bar')
    ax.scatter([ev.name], [df.loc[ev.name, 'Close']], color='#E91E63', s=100, zorder=5, edgecolors='black')

    ax.set_title(f'Triangle Breakout: {ev["triangle_pattern"]} on {ev.name.strftime("%Y-%m-%d")}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Price ($)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

### 4c. Price Channels — Visualized

In [ ]:
ch_events = ch_df[ch_df['channel_pattern'].notna()]
print(f'Total channel events: {len(ch_events)}')
print(f'\nBreakdown:')
print(ch_events['channel_pattern'].value_counts())

if len(ch_events) > 0:
    ev = ch_events.iloc[len(ch_events) // 3]
    ev_idx = df.index.get_loc(ev.name)
    lo, hi = max(0, ev_idx - 60), min(len(df), ev_idx + 10)
    slc = df.iloc[lo:hi]

    fig, ax = plt.subplots(figsize=(14, 6))
    for _, row in slc.iterrows():
        c = '#26a69a' if row['Close'] >= row['Open'] else '#ef5350'
        ax.plot([row.name, row.name], [row['Low'], row['High']], color=c, lw=0.7)
        ax.plot([row.name, row.name], [row['Open'], row['Close']], color=c, lw=3)

    win = 50
    wslc = df.iloc[ev_idx - win:ev_idx]
    x = np.arange(win)
    hc = np.polyfit(x, wslc['High'].values, 1)
    lc = np.polyfit(x, wslc['Low'].values, 1)
    ax.plot(wslc.index, np.polyval(hc, x), color='#F44336', lw=2, label='Upper trendline')
    ax.plot(wslc.index, np.polyval(lc, x), color='#2196F3', lw=2, label='Lower trendline')
    ax.fill_between(wslc.index, np.polyval(lc, x), np.polyval(hc, x), alpha=0.08, color='#FF9800')

    ax.axvline(ev.name, color='#E91E63', lw=2, ls='--')
    ax.scatter([ev.name], [df.loc[ev.name, 'Close']], color='#E91E63', s=100, zorder=5,
               edgecolors='black', label=f'Event: {ev["channel_pattern"]}')

    ax.set_title(f'Channel Detection: {ev["channel_pattern"]} on {ev.name.strftime("%Y-%m-%d")}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Price ($)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

### 4d. Multiple Tops & Bottoms — Visualized

In [ ]:
mtb_events = mtb_df[mtb_df['multiple_top_bottom_pattern'].notna()]
print(f'Total multi top/bottom events: {len(mtb_events)}')
print(f'\nBreakdown:')
print(mtb_events['multiple_top_bottom_pattern'].value_counts())

if len(mtb_events) > 0:
    # Show a few examples on the price chart
    tops = mtb_events[mtb_events['multiple_top_bottom_pattern'] == 'multiple_top']
    bots = mtb_events[mtb_events['multiple_top_bottom_pattern'] == 'multiple_bottom']

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(df.index, df['Close'], color='#333', lw=0.6)
    if len(tops) > 0:
        ax.scatter(tops.index, df.loc[tops.index, 'Close'], color='#F44336', s=12, alpha=0.6,
                   label=f'Multiple Top ({len(tops)})', zorder=3)
    if len(bots) > 0:
        ax.scatter(bots.index, df.loc[bots.index, 'Close'], color='#4CAF50', s=12, alpha=0.6,
                   label=f'Multiple Bottom ({len(bots)})', zorder=3)
    ax.set_title('Multiple Tops (red) & Bottoms (green) on SPY', fontsize=13, fontweight='bold')
    ax.set_ylabel('Price ($)')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

---
## 5. Combined Event Overlay

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df.index, df['Close'], color='#333', lw=0.6, label='SPY Close')

detectors = {
    'S/R': ('#2196F3', sr),
    'Triangle': ('#E91E63', tri),
    'Channel': ('#FF9800', ch),
    'Multi T/B': ('#4CAF50', mtb),
}
for label, (color, mask) in detectors.items():
    idx = df.index[mask]
    if len(idx) > 0:
        ax.scatter(idx, df.loc[idx, 'Close'], color=color, s=6, alpha=0.6, label=f'{label} ({mask.sum()})')

ax.set_title(f'All Detected Events on SPY — Combined Rate: {combined.mean()*100:.1f}%',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

---
## 6. Event Rate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Pie chart
labels = ['S/R', 'Triangles', 'Channels', 'Multi T/B']
counts = [sr.sum(), tri.sum(), ch.sum(), mtb.sum()]
clrs = ['#2196F3', '#E91E63', '#FF9800', '#4CAF50']
axes[0].pie(counts, labels=labels, colors=clrs, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Event Distribution', fontweight='bold')

# Monthly rate
monthly = combined.resample('ME').mean() * 100
bar_colors = ['#4CAF50' if 10 <= v <= 30 else '#F44336' for v in monthly]
axes[1].bar(monthly.index, monthly.values, width=25, color=bar_colors, alpha=0.8)
axes[1].axhline(10, color='#2e7d32', ls='--', lw=0.8)
axes[1].axhline(30, color='#c62828', ls='--', lw=0.8)
axes[1].set_title('Monthly Event Rate (%)', fontweight='bold')
axes[1].set_ylabel('%')

# Bar comparison
det_names = ['S/R', 'Tri', 'Chan', 'MTB', 'COMBINED']
det_rates = [sr.mean()*100, tri.mean()*100, ch.mean()*100, mtb.mean()*100, combined.mean()*100]
bar_c = ['#2196F3', '#E91E63', '#FF9800', '#4CAF50', '#5C6BC0']
axes[2].barh(det_names, det_rates, color=bar_c)
axes[2].axvspan(10, 30, alpha=0.1, color='green')
for i, v in enumerate(det_rates):
    axes[2].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
axes[2].set_title('Detection Rates', fontweight='bold')
axes[2].set_xlabel('%')

plt.tight_layout()
plt.show()

---
## 7. Spot-Check: Forward Returns After Events

In [ ]:
# Sample 20 events and check what happened in the next 10 days
event_dates = df.index[combined]
np.random.seed(42)
step = max(1, len(event_dates) // 20)
sampled = [event_dates[i * step] for i in range(min(20, len(event_dates)))]

results = []
for dt in sampled:
    pos = df.index.get_loc(dt)
    fwd = min(pos + 10, len(df) - 1)
    ret = (df.iloc[fwd]['Close'] - df.loc[dt, 'Close']) / df.loc[dt, 'Close'] * 100

    pattern = []
    if sr.get(dt, False): pattern.append('S/R')
    if tri.get(dt, False): pattern.append('Tri')
    if ch.get(dt, False): pattern.append('Chan')
    if mtb.get(dt, False): pattern.append('MTB')

    results.append({'date': dt.strftime('%Y-%m-%d'), 'pattern': ', '.join(pattern),
                    'close': df.loc[dt, 'Close'], '10d_return': ret})

results_df = pd.DataFrame(results)
print(f'Mean 10d return after event: {results_df["10d_return"].mean():+.2f}%')
print(f'Positive:  {(results_df["10d_return"] > 0).sum()} / {len(results_df)}')
print()
results_df

In [ ]:
# Visualize forward returns
fig, ax = plt.subplots(figsize=(14, 5))
bar_colors = ['#4CAF50' if r > 0 else '#F44336' for r in results_df['10d_return']]
ax.bar(range(len(results_df)), results_df['10d_return'], color=bar_colors, alpha=0.8,
       edgecolor='#333', linewidth=0.5)
ax.axhline(0, color='#333', lw=0.8)

for i, row in results_df.iterrows():
    ax.text(i, row['10d_return'] + (0.1 if row['10d_return'] >= 0 else -0.15),
            f"{row['pattern']}\n{row['10d_return']:+.1f}%",
            ha='center', va='bottom' if row['10d_return'] >= 0 else 'top', fontsize=6)

ax.set_xticks(range(len(results_df)))
ax.set_xticklabels([d[:7] for d in results_df['date']], rotation=45, fontsize=7)
ax.set_ylabel('10-Day Forward Return (%)')
ax.set_title('Spot-Check: What Happened After Each Sampled Event?', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Regime Concept Preview

This is a **volatility-based proxy** for what the HMM will do in Phase 2. It shows how different regimes have very different return distributions.

In [ ]:
# Volatility-based regime proxy
returns = df['Close'].pct_change()
vol = returns.rolling(50).std() * np.sqrt(252)

regime = pd.Series('Sideways', index=df.index)
regime[vol <= vol.quantile(0.3)] = 'Bull'
regime[vol > vol.quantile(0.7)] = 'Bear'

cmap = {'Bull': '#4CAF50', 'Bear': '#F44336', 'Sideways': '#FF9800'}

fig, axes = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={'height_ratios': [3, 1.2]})

ax = axes[0]
ax.plot(df.index, df['Close'], color='#333', lw=0.7)
for r, color in cmap.items():
    mask = regime == r
    segments = mask.astype(int).diff().fillna(0)
    starts = df.index[segments == 1]
    ends = df.index[segments == -1]
    if mask.iloc[0]: starts = starts.insert(0, df.index[0])
    if mask.iloc[-1]: ends = ends.append(pd.DatetimeIndex([df.index[-1]]))
    for s, e in zip(starts, ends):
        ax.axvspan(s, e, alpha=0.12, color=color)

patches = [mpatches.Patch(color=c, alpha=0.3, label=r) for r, c in cmap.items()]
ax.legend(handles=patches, fontsize=9, loc='upper left')
ax.set_title('SPY with Illustrative Market Regimes', fontsize=13, fontweight='bold')
ax.set_ylabel('Price ($)')

ax2 = axes[1]
ax2.fill_between(df.index, vol * 100, alpha=0.4, color='#7986CB')
ax2.axhline(vol.quantile(0.3) * 100, color='#4CAF50', ls='--', lw=0.8, label='30th pctl (Bull/Sideways)')
ax2.axhline(vol.quantile(0.7) * 100, color='#F44336', ls='--', lw=0.8, label='70th pctl (Sideways/Bear)')
ax2.set_ylabel('Ann. Vol (%)')
ax2.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Return distributions by regime
fig, ax = plt.subplots(figsize=(12, 5))

returns_clean = returns.dropna()
vol_clean = vol.reindex(returns_clean.index)

bull_r = returns_clean[vol_clean <= vol_clean.quantile(0.3)]
bear_r = returns_clean[vol_clean > vol_clean.quantile(0.7)]
side_r = returns_clean[(vol_clean > vol_clean.quantile(0.3)) & (vol_clean <= vol_clean.quantile(0.7))]

bins = np.linspace(-0.06, 0.06, 80)
ax.hist(bull_r, bins=bins, alpha=0.5, color='#4CAF50', label=f'Bull (n={len(bull_r)}, mean={bull_r.mean()*100:+.2f}%)', density=True)
ax.hist(bear_r, bins=bins, alpha=0.5, color='#F44336', label=f'Bear (n={len(bear_r)}, mean={bear_r.mean()*100:+.2f}%)', density=True)
ax.hist(side_r, bins=bins, alpha=0.3, color='#FF9800', label=f'Sideways (n={len(side_r)})', density=True)

ax.set_title('Daily Return Distributions by Regime — Why Regime Matters', fontsize=12, fontweight='bold')
ax.set_xlabel('Daily Return')
ax.set_ylabel('Density')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'Bull regime:  mean={bull_r.mean()*100:+.3f}%, std={bull_r.std()*100:.3f}%')
print(f'Bear regime:  mean={bear_r.mean()*100:+.3f}%, std={bear_r.std()*100:.3f}%')
print(f'Sideways:     mean={side_r.mean()*100:+.3f}%, std={side_r.std()*100:.3f}%')

---
## 9. Run the Full Scanner

This is how the system is used end-to-end: one function call that runs all detectors and returns the enriched DataFrame.

In [ ]:
from src.patterns.scanner import scan_all_patterns, get_events

# Run everything
enriched = scan_all_patterns(df)

print(f'Total bars:  {len(enriched)}')
print(f'Event bars:  {enriched["has_event"].sum()} ({enriched["has_event"].mean()*100:.1f}%)')
print()
print('Pattern breakdown:')
print(f'  Near support:     {enriched["near_support"].sum()}')
print(f'  Near resistance:  {enriched["near_resistance"].sum()}')
print(f'  Triangles:        {enriched["triangle_pattern"].notna().sum()}')
print(f'  Multi top/bottom: {enriched["multiple_top_bottom_pattern"].notna().sum()}')
print(f'  Channels:         {enriched["channel_pattern"].notna().sum()}')
print()
print('Columns added:')
print([c for c in enriched.columns if c not in df.columns])

In [ ]:
# Show event rows
events = get_events(df)
events[['Close', 'support', 'resistance', 'near_support', 'near_resistance',
         'triangle_pattern', 'channel_pattern', 'multiple_top_bottom_pattern']].head(20)

---
## 10. Interactive Exploration

Use the cells below to explore any date range or event you're interested in.

In [ ]:
def explore_event(event_date_str, context_bars=40):
    """Visualize an event with surrounding context.

    Usage: explore_event('2020-03-23')  or  explore_event('2022-01-20')
    """
    dt = pd.Timestamp(event_date_str)
    pos = df.index.get_indexer([dt], method='nearest')[0]
    lo = max(0, pos - context_bars)
    hi = min(len(df), pos + context_bars + 1)
    slc = df.iloc[lo:hi]
    actual_dt = df.index[pos]

    fig, ax = plt.subplots(figsize=(14, 6))
    for _, row in slc.iterrows():
        c = '#26a69a' if row['Close'] >= row['Open'] else '#ef5350'
        ax.plot([row.name, row.name], [row['Low'], row['High']], color=c, lw=0.7)
        ax.plot([row.name, row.name], [row['Open'], row['Close']], color=c, lw=3)

    ax.axvline(actual_dt, color='#E91E63', lw=2, ls='--', label=f'Selected: {actual_dt.date()}')

    # Show S/R if available
    if actual_dt in sr_df.index:
        s = sr_df.loc[actual_dt, 'support']
        r = sr_df.loc[actual_dt, 'resistance']
        if pd.notna(s): ax.axhline(s, color='#2196F3', lw=0.8, ls=':', label=f'Support: ${s:.2f}')
        if pd.notna(r): ax.axhline(r, color='#F44336', lw=0.8, ls=':', label=f'Resistance: ${r:.2f}')

    # Print what fired
    signals = []
    if sr.get(actual_dt, False): signals.append('S/R')
    if tri.get(actual_dt, False): signals.append(tri_df.loc[actual_dt, 'triangle_pattern'])
    if ch.get(actual_dt, False): signals.append(ch_df.loc[actual_dt, 'channel_pattern'])
    if mtb.get(actual_dt, False): signals.append(mtb_df.loc[actual_dt, 'multiple_top_bottom_pattern'])

    title = f'{actual_dt.date()} — {", ".join(signals) if signals else "No event"}'
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Price ($)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

# Try it:
explore_event('2020-03-23')  # COVID crash bottom

In [ ]:
# Explore another date — change this to any date you want!
explore_event('2022-01-20')

In [ ]:
# List all events in a specific year
year = 2023
year_events = events.loc[str(year)]
print(f'Events in {year}: {len(year_events)}')
year_events[['Close', 'triangle_pattern', 'channel_pattern', 'multiple_top_bottom_pattern',
              'near_support', 'near_resistance']].head(30)

---
## Summary

| Component | Status | Key Metric |
|:--|:--|:--|
| Data Pipeline | Done | 4,024 bars (2010-2025) |
| ATR Utility | Done | Adaptive volatility measure |
| Support/Resistance | Done | ~19% event rate |
| Triangle Breakouts | Done | ~2% event rate |
| Price Channels | Done | ~6% event rate |
| Multiple Tops/Bottoms | Done | ~3.5% event rate |
| **Combined Rate** | **Done** | **~27% (target 10-30%)** |
| Feature Engineering | Next | — |
| HMM Regime Detection | Next | — |
| Signal Model (XGBoost) | Planned | — |
| Walk-Forward Backtest | Planned | — |